In [69]:
import numpy as np
import pandas as pd
import re
import string
import pickle

In [70]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation,'')
    return text

In [71]:
with open('../static/model/model.pickle','rb') as file:
    model =  pickle.load(file)

In [72]:
with open('../static/model/corpora/stopwords/english','r') as file:
    sw = file.read().splitlines()

In [73]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [74]:
vocab = pd.read_csv('../static/model/vocabulary.txt',header=None)
tokens = vocab[0].tolist()

In [75]:
def preprocessing(text):
    data = pd.DataFrame([text],columns=['tweet'])
    data['tweet'] = data['tweet'].apply(lambda x: " ".join(x.lower() for x in x.split()))
    data['tweet'] = data['tweet'].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*','',x,flags=re.MULTILINE) for x in x.split()))
    data['tweet'] = data['tweet'].apply(remove_punctuations)
    data['tweet'] = data['tweet'].str.replace(r'\d+','',regex=True)
    data['tweet'] =  data['tweet'].apply(lambda x: " ".join(x for x in x.split() if x not in sw))
    data['tweet'] =  data['tweet'].apply(lambda x: " ".join(ps.stem(x) for x in x.split()))
    return data['tweet']

In [76]:
def vectorization(ds,vocabulary):
    vectorized_list = []

    for sentence in ds:
        sentence_list = np.zeros(len(vocabulary))

        for i in range(len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_list[i] = 1
                
        vectorized_list.append(sentence_list)
    vectorized_list_new =  np.asarray(vectorized_list,dtype=np.float32)
    return vectorized_list_new

In [81]:
def get_prediction(vectorized_text):
    prediction = model.predict(vecctorize_text)
    if prediction == 1:
        return 'negative'
    else:
        return 'positive'

In [82]:
text = 'good product, I love it'
preprocess_text = preprocessing(text)
vecctorize_text = vectorization(preprocess_text,tokens)
prediction = get_prediction(vecctorize_text)
prediction

'positive'